# 09-1. LoRA · QLoRA Fine-Tuning

Qwen2.5-1.5B-Instruct 를 **LoRA** / **QLoRA** 두 방식으로 파인튜닝하고 자원 사용량을 비교

| 구분 | 베이스 모델 로딩 | 학습 대상 |
|---|---|---|
| LoRA | fp16 그대로 | LoRA 어댑터만 |
| QLoRA | 4bit(NF4)로 양자화 | LoRA 어댑터만 (베이스는 4bit 동결) |

> **두 방식의 코드 차이는 `BitsAndBytesConfig` 를 주느냐 마느냐 한 줄뿐** — 학습 루프는 완전히 동일

**산출물** (다음 노트북들이 입력으로 사용)
- `merged fp16 모델` → HuggingFace Hub (private)
- `metrics_train.json` → Google Drive

**런타임: GPU (T4 이상)**

## 0. 환경 설정

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets huggingface_hub

In [ ]:
import os, json, time, gc, torch

MODEL_ID   = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID = "beomi/KoAlpaca-v1.1a"
N_TRAIN, N_EVAL = 1000, 100
MAX_LEN = 512

HF_USER    = "본인_HF_아이디"          # TODO
MERGED_REPO = f"{HF_USER}/qwen2.5-1.5b-koalpaca-merged"

SAVE_DIR = "/content/drive/MyDrive/ktb/09_llm_opt"   # Drive 저장 경로

# T4 는 bf16 미지원 -> fp16 으로 자동 전환
BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", DTYPE)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
os.makedirs(SAVE_DIR, exist_ok=True)

from huggingface_hub import login
login(userdata.get('HF_TOKEN'))   # Colab 좌측 열쇠 아이콘에 HF_TOKEN 등록 (write 권한)

## 1. 데이터 준비

Instruction 데이터를 Qwen 의 **chat template** 에 맞춰 토크나이즈

핵심은 **label masking** — 프롬프트(질문) 구간은 `-100` 으로 마스킹해서 loss 에서 제외하고,
**답변 토큰에 대해서만 loss 를 계산**해야 함. 이걸 안 하면 모델이 "질문을 생성하는 법"까지 학습함

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

raw = load_dataset(DATASET_ID, split="train").shuffle(seed=42)
raw = raw.select(range(N_TRAIN + N_EVAL))

def build(example):
    # 1) 프롬프트 구간: chat template + generation prompt
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": example["instruction"]}],
        tokenize=False, add_generation_prompt=True,
    )
    answer = example["output"] + tokenizer.eos_token

    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    a_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]

    input_ids = (p_ids + a_ids)[:MAX_LEN]
    # 2) 프롬프트 구간은 -100 으로 마스킹
    labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]
    return {"input_ids": input_ids, "labels": labels,
            "attention_mask": [1] * len(input_ids)}

ds = raw.map(build, remove_columns=raw.column_names)
ds = ds.train_test_split(test_size=N_EVAL, seed=42)
train_ds, eval_ds = ds["train"], ds["test"]

print(train_ds)
print(tokenizer.decode(train_ds[0]["input_ids"]))

In [ ]:
# 이후 노트북(PTQ 캘리브레이션 / perplexity 평가)에서 재사용하도록 eval 텍스트 저장
eval_texts = [tokenizer.decode(x, skip_special_tokens=False) for x in eval_ds["input_ids"]]
with open(f"{SAVE_DIR}/eval_texts.json", "w") as f:
    json.dump(eval_texts, f, ensure_ascii=False)
len(eval_texts)

## 2. 공통 학습 함수

`use_4bit` 플래그 하나로 LoRA / QLoRA 가 갈림

In [ ]:
from transformers import (AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

LORA_CFG = dict(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
)

def load_base(use_4bit: bool):
    kwargs = dict(dtype=DTYPE, device_map={"": 0})
    if use_4bit:
        # === QLoRA 의 전부: 베이스를 NF4 4bit 로 로딩 ===
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",          # 정규분포에 최적화된 4bit 자료형
            bnb_4bit_use_double_quant=True,     # scale 자체도 한 번 더 양자화
            bnb_4bit_compute_dtype=DTYPE,       # 연산은 16bit 로 역양자화해서 수행
        )
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)


def run(tag: str, use_4bit: bool, epochs: float = 1.0):
    torch.cuda.empty_cache(); gc.collect()
    torch.cuda.reset_peak_memory_stats()

    model = load_base(use_4bit)
    if use_4bit:
        # 4bit 모듈 dtype 정리 + gradient checkpointing 호환 처리
        model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False
    model.enable_input_require_grads()

    model = get_peft_model(model, LoraConfig(**LORA_CFG))
    model.print_trainable_parameters()

    args = TrainingArguments(
        output_dir=f"/content/out_{tag}",
        num_train_epochs=epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        optim="paged_adamw_8bit" if use_4bit else "adamw_torch",
        bf16=BF16, fp16=not BF16,
        report_to="none",
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True,
                                             label_pad_token_id=-100),
    )

    t0 = time.time()
    hist = trainer.train()
    elapsed = time.time() - t0
    ev = trainer.evaluate()

    m = {
        "tag": tag,
        "peak_vram_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "train_time_sec": round(elapsed, 1),
        "train_loss": round(hist.training_loss, 4),
        "eval_loss": round(ev["eval_loss"], 4),
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    }
    model.save_pretrained(f"/content/adapter_{tag}")

    del trainer, model
    torch.cuda.empty_cache(); gc.collect()
    return m

### 2-1. LoRA 학습

In [ ]:
m_lora = run("lora", use_4bit=False)
m_lora

### 2-2. QLoRA 학습

> 런타임 재시작 없이 이어서 돌리면 VRAM 측정이 오염될 수 있음 — 값이 이상하면 런타임 재시작 후 QLoRA 셀만 다시 실행

In [ ]:
m_qlora = run("qlora", use_4bit=True)
m_qlora

## 3. LoRA vs QLoRA 비교

In [ ]:
import pandas as pd
df = pd.DataFrame([m_lora, m_qlora]).set_index("tag")
df.to_csv(f"{SAVE_DIR}/metrics_train.csv")
with open(f"{SAVE_DIR}/metrics_train.json", "w") as f:
    json.dump([m_lora, m_qlora], f, indent=2)
df

**해석 포인트**
- `peak_vram_GB` : QLoRA 가 확연히 낮아야 정상 (베이스 가중치가 4bit)
- `train_time_sec` : QLoRA 가 **더 느릴 수 있음** — 매 연산마다 4bit → 16bit 역양자화 비용이 붙기 때문. 메모리를 시간과 맞바꾼 것
- `eval_loss` : 차이가 크지 않아야 정상. 4bit 로 줄여도 성능이 거의 유지된다는 게 QLoRA 논문의 핵심 주장

## 4. 어댑터 병합 → merged fp16 모델

**주의: QLoRA 어댑터는 4bit 베이스 위에서 학습됐지만, 병합은 반드시 fp16 베이스에 해야 함**

4bit 모델에 `merge_and_unload()` 를 그대로 부르면 양자화된 가중치에 어댑터를 더하게 돼서
값이 깨지거나 에러가 남. 그래서 **베이스를 fp16 으로 다시 로딩한 뒤** 어댑터를 얹어 병합함

(어댑터 자체는 애초에 fp16 으로 학습된 별도 파라미터라 fp16 베이스에 그대로 붙일 수 있음)

In [ ]:
from peft import PeftModel

MERGE_FROM = "qlora"   # 병합할 어댑터 선택

torch.cuda.empty_cache(); gc.collect()

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16,
                                            device_map={"": 0})   # ← 4bit 아님
model = PeftModel.from_pretrained(base, f"/content/adapter_{MERGE_FROM}")
merged = model.merge_and_unload()
merged.config.use_cache = True

merged.save_pretrained("/content/merged", safe_serialization=True)
tokenizer.save_pretrained("/content/merged")
!du -sh /content/merged

In [ ]:
# 병합 모델 동작 확인
from transformers import pipeline
pipe = pipeline("text-generation", model=merged, tokenizer=tokenizer)
msg = [{"role": "user", "content": "LoRA와 QLoRA의 차이를 3줄로 설명해줘"}]
print(pipe(tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True),
           max_new_tokens=200, do_sample=False)[0]["generated_text"])

In [ ]:
# Hub 업로드 -> 02, 03 노트북이 여기서 받아감
merged.push_to_hub(MERGED_REPO, private=True)
tokenizer.push_to_hub(MERGED_REPO, private=True)
print("pushed:", MERGED_REPO)

In [ ]:
# 어댑터도 Drive 에 백업 (수십 MB)
!cp -r /content/adapter_lora /content/adapter_qlora "{SAVE_DIR}"/
!ls "{SAVE_DIR}"